In [ ]:
#cell1
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate scikit-learn mlflow

import torch
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.5 MB/s eta 0:00:00
Tesla T4


In [ ]:
#cell-2
import shutil, os

os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/data/results', exist_ok=True)

shutil.copy('/content/drive/MyDrive/contract_risk/data/contract_train.jsonl', '/content/data/processed/')
shutil.copy('/content/drive/MyDrive/contract_risk/data/contract_val.jsonl',   '/content/data/processed/')
shutil.copy('/content/drive/MyDrive/contract_risk/data/contract_test.jsonl',  '/content/data/processed/')

print("Data copied!")

Data copied!


In [ ]:
#cell-3
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import mlflow

MODEL_NAME  = "bert-base-uncased"
MAX_LENGTH  = 256
BATCH_SIZE  = 32
NUM_EPOCHS  = 3
NUM_LABELS  = 8
DATA_DIR    = Path("/content/data/processed")
RESULTS_DIR = Path("/content/data/results")
RESULTS_DIR.mkdir(exist_ok=True)

LABEL_MAP = {
    0: "Limitation of Liability",
    1: "Unilateral Termination",
    2: "Unilateral Change",
    3: "Content Removal",
    4: "Contract by Using",
    5: "Choice of Law",
    6: "Choice of Venue",
    7: "Forced Arbitration",
}

def load_jsonl(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            texts.append(ex["input"])
            output = ex["output"]
            if output.startswith("VERDICT: FAIR"):
                labels.append([0.0] * NUM_LABELS)
            else:
                row = [0.0] * NUM_LABELS
                for lid, lname in LABEL_MAP.items():
                    if lname in output:
                        row[lid] = 1.0
                labels.append(row)
    return texts, labels

class ClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds    = (logits > 0.0).astype(int)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    return {"macro_f1": macro_f1, "micro_f1": micro_f1}

# ── Run ───────────────────────────────────────────────────────────────────────

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading data...")
train_texts, train_labels = load_jsonl(DATA_DIR / "contract_train.jsonl")
val_texts,   val_labels   = load_jsonl(DATA_DIR / "contract_val.jsonl")
test_texts,  test_labels  = load_jsonl(DATA_DIR / "contract_test.jsonl")
print(f"  Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}")

print("Tokenising...")
train_ds = ClauseDataset(train_texts, train_labels, tokenizer)
val_ds   = ClauseDataset(val_texts,   val_labels,   tokenizer)
test_ds  = ClauseDataset(test_texts,  test_labels,  tokenizer)

print("Loading BERT-base...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

training_args = TrainingArguments(
    output_dir="/content/data/bert_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("\nTraining BERT-base on T4...")
trainer.train()

print("\nEvaluating on test set...")
preds_output = trainer.predict(test_ds)
preds  = (preds_output.predictions > 0.0).astype(int)
labels = np.array(test_labels)

macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)

print(f"\n{'='*50}")
print("RESULTS — BERT-base fine-tuned")
print(f"{'='*50}")
print(f"Macro F1 : {macro_f1:.4f}")
print(f"Micro F1 : {micro_f1:.4f}")
print("\nPer-class report:")
print(classification_report(
    labels, preds,
    target_names=[LABEL_MAP[i] for i in range(8)],
    zero_division=0,
))

# Save results
with open(RESULTS_DIR / "results.md", "w") as f:
    f.write("# Baseline Results\n\n")
    f.write("| Model | Macro F1 |\n")
    f.write("|---|---|\n")
    f.write(f"| Logistic Regression + TF-IDF | 0.1603 |\n")
    f.write(f"| BERT-base (fine-tuned) | {macro_f1:.4f} |\n")
    f.write("| Mistral 7B prompt-only | TBD |\n")
    f.write("| Mistral 7B QLoRA fine-tuned | TBD |\n")

print("\nResults saved!")

Loading tokenizer...
Loading data...
  Train: 5,475 | Val: 2,252 | Test: 1,587
Tokenising...
Loading BERT-base...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training BERT-base on T4...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on test set...



RESULTS — BERT-base fine-tuned
Macro F1 : 0.0000
Micro F1 : 0.0000

Per-class report:
                         precision    recall  f1-score   support

Limitation of Liability       0.00      0.00      0.00        38
 Unilateral Termination       0.00      0.00      0.00        38
      Unilateral Change       0.00      0.00      0.00        38
        Content Removal       0.00      0.00      0.00        13
      Contract by Using       0.00      0.00      0.00        23
          Choice of Law       0.00      0.00      0.00        13
        Choice of Venue       0.00      0.00      0.00        16
     Forced Arbitration       0.00      0.00      0.00         7

              micro avg       0.00      0.00      0.00       186
              macro avg       0.00      0.00      0.00       186
           weighted avg       0.00      0.00      0.00       186
            samples avg       0.00      0.00      0.00       186


Results saved!


In [ ]:
#cell-4
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME  = "bert-base-uncased"
MAX_LENGTH  = 256
BATCH_SIZE  = 32
NUM_EPOCHS  = 5
NUM_LABELS  = 8
DATA_DIR    = Path("/content/data/processed")
RESULTS_DIR = Path("/content/data/results")

LABEL_MAP = {
    0: "Limitation of Liability",
    1: "Unilateral Termination",
    2: "Unilateral Change",
    3: "Content Removal",
    4: "Contract by Using",
    5: "Choice of Law",
    6: "Choice of Venue",
    7: "Forced Arbitration",
}

def load_jsonl(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            texts.append(ex["input"])
            output = ex["output"]
            if output.startswith("VERDICT: FAIR"):
                labels.append([0.0] * NUM_LABELS)
            else:
                row = [0.0] * NUM_LABELS
                for lid, lname in LABEL_MAP.items():
                    if lname in output:
                        row[lid] = 1.0
                labels.append(row)
    return texts, labels

class ClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=MAX_LENGTH, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# ── Weighted loss: unfair classes get 8x more weight ─────────────────────────
class WeightedMultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        # pos_weight > 1 = penalise false negatives on minority classes
        pos_weight = torch.ones(NUM_LABELS, device=logits.device) * 8.0
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=pos_weight
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Lower threshold to 0.3 to catch more unfair clauses
    preds    = (torch.sigmoid(torch.tensor(logits)) > 0.3).numpy().astype(int)
    macro_f1 = f1_score(labels, preds, average="macro",  zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro",  zero_division=0)
    return {"macro_f1": macro_f1, "micro_f1": micro_f1}

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading tokenizer + data...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts, train_labels = load_jsonl(DATA_DIR / "contract_train.jsonl")
val_texts,   val_labels   = load_jsonl(DATA_DIR / "contract_val.jsonl")
test_texts,  test_labels  = load_jsonl(DATA_DIR / "contract_test.jsonl")

train_ds = ClauseDataset(train_texts, train_labels, tokenizer)
val_ds   = ClauseDataset(val_texts,   val_labels,   tokenizer)
test_ds  = ClauseDataset(test_texts,  test_labels,  tokenizer)

print("Loading BERT-base...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

training_args = TrainingArguments(
    output_dir="/content/data/bert_checkpoints_v2",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = WeightedMultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("\nTraining with weighted loss...")
trainer.train()

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\nEvaluating on test set...")
preds_output = trainer.predict(test_ds)
logits = preds_output.predictions
preds  = (torch.sigmoid(torch.tensor(logits)) > 0.3).numpy().astype(int)
labels = np.array(test_labels)

macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)

print(f"\n{'='*50}")
print("RESULTS — BERT-base with weighted loss")
print(f"{'='*50}")
print(f"Macro F1 : {macro_f1:.4f}")
print(f"Micro F1 : {micro_f1:.4f}")
print("\nPer-class report:")
print(classification_report(
    labels, preds,
    target_names=[LABEL_MAP[i] for i in range(8)],
    zero_division=0,
))

# Save
with open(RESULTS_DIR / "results.md", "w") as f:
    f.write("# Baseline Results\n\n")
    f.write("| Model | Macro F1 |\n|---|---|\n")
    f.write("| Logistic Regression + TF-IDF | 0.1603 |\n")
    f.write(f"| BERT-base + weighted loss | {macro_f1:.4f} |\n")
    f.write("| Mistral 7B prompt-only | TBD |\n")
    f.write("| Mistral 7B QLoRA fine-tuned | TBD |\n")
print("Results saved!")

Loading tokenizer + data...
Loading BERT-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training with weighted loss...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1
1,0.280509,0.196367,0.333274,0.330947
2,0.154066,0.121703,0.393110,0.400706
3,0.094294,0.113568,0.410901,0.410210
4,0.072012,0.101792,0.495877,0.514552
5,0.057594,0.099866,0.521872,0.538835


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on test set...



RESULTS — BERT-base with weighted loss
Macro F1 : 0.5492
Micro F1 : 0.5632

Per-class report:
                         precision    recall  f1-score   support

Limitation of Liability       0.44      0.84      0.58        38
 Unilateral Termination       0.51      0.92      0.65        38
      Unilateral Change       0.43      0.87      0.57        38
        Content Removal       0.23      1.00      0.38        13
      Contract by Using       0.71      0.87      0.78        23
          Choice of Law       0.41      1.00      0.58        13
        Choice of Venue       0.41      0.94      0.57        16
     Forced Arbitration       0.17      0.86      0.28         7

              micro avg       0.41      0.90      0.56       186
              macro avg       0.41      0.91      0.55       186
           weighted avg       0.46      0.90      0.59       186
            samples avg       0.06      0.10      0.07       186

Results saved!


In [ ]:
#cell-5
import shutil
shutil.copy('/content/data/results/results.md',
            '/content/drive/MyDrive/contract_risk/results.md')
print("Saved to Drive!")

Saved to Drive!


In [ ]:
#cell-6
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
Tesla T4


In [ ]:
#cell-7
import shutil, os
os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/data/results', exist_ok=True)
shutil.copy('/content/drive/MyDrive/contract_risk/data/contract_test.jsonl', '/content/data/processed/')
print("Data copied!")

Data copied!


In [ ]:
#cell- 8
!pip install -q -U bitsandbytes>=0.46.1
import importlib
import bitsandbytes
importlib.reload(bitsandbytes)
print("bitsandbytes version:", bitsandbytes.__version__)

bitsandbytes version: 0.49.2


In [ ]:
#cell-9
!pip install -q -U bitsandbytes

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/data/results', exist_ok=True)
shutil.copy('/content/drive/MyDrive/contract_risk/data/contract_test.jsonl', '/content/data/processed/')
print("Data copied!")

Mounted at /content/drive
Data copied!


In [3]:
!pip install -q -U bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading Mistral 7B in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
Loading Mistral 7B in 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded!


In [4]:
SYSTEM_PROMPT = """You are a legal contract analyst.
For each clause, respond ONLY in this exact format:
VERDICT: FAIR or UNFAIR
RISK TYPE: (comma-separated from: Limitation of Liability, Unilateral Termination, Unilateral Change, Content Removal, Contract by Using, Choice of Law, Choice of Venue, Forced Arbitration) or None if FAIR

No explanation. Just those two lines."""

import json
from pathlib import Path

# Load 5 test examples
test_texts = []
with open('/content/data/processed/contract_test.jsonl', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        test_texts.append(json.loads(line)['input'])

# Run 5 examples and print raw responses
for i, clause in enumerate(test_texts):
    prompt = f"[INST] {SYSTEM_PROMPT}\n\nClause:\n{clause[:600]} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=700).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n--- Example {i+1} ---")
    print(f"CLAUSE: {clause[:80]}...")
    print(f"RESPONSE: {repr(response)}")


--- Example 1 ---
CLAUSE: last updated date : may 15 , 2017...
RESPONSE: 'VERDICT: FAIR\nRISK TYPE: None'

--- Example 2 ---
CLAUSE: academia , inc. ( `` academia.edu '' or `` we '' ) offers a social networking se...
RESPONSE: 'VERDICT: FAIR\nRISK TYPE: None'

--- Example 3 ---
CLAUSE: please read carefully the following terms and conditions ( `` terms '' ) and our...
RESPONSE: 'VERDICT: FAIR\nRISK TYPE: Choice of Law, Choice of Venue, None (since it is fair)'

--- Example 4 ---
CLAUSE: these terms govern your access to and use of the site , services and collective ...
RESPONSE: 'VERDICT: FAIR\nRISK TYPE: Choice of Law, Choice of Venue, Contract by Using'

--- Example 5 ---
CLAUSE: arbitration notice : unless you opt out of arbitration within 30 days of the dat...
RESPONSE: 'VERDICT: FAIR\nRISK TYPE: Forced Arbitration, Unilateral Change (opt-out required within 30 days)'


In [5]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

NUM_SAMPLES = 200
DATA_DIR    = Path("/content/data/processed")
RESULTS_DIR = Path("/content/data/results")
RESULTS_DIR.mkdir(exist_ok=True)

LABEL_MAP = {
    0: "Limitation of Liability",
    1: "Unilateral Termination",
    2: "Unilateral Change",
    3: "Content Removal",
    4: "Contract by Using",
    5: "Choice of Law",
    6: "Choice of Venue",
    7: "Forced Arbitration",
}

def load_jsonl(path, max_examples=None):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_examples and i >= max_examples:
                break
            ex = json.loads(line)
            texts.append(ex["input"])
            output = ex["output"]
            if output.startswith("VERDICT: FAIR"):
                labels.append([0.0] * 8)
            else:
                row = [0.0] * 8
                for lid, lname in LABEL_MAP.items():
                    if lname in output:
                        row[lid] = 1.0
                labels.append(row)
    return texts, labels

def parse_response(response: str) -> list:
    """
    Parse Mistral output into binary label vector.
    Trust risk types over verdict — if risk types are listed, mark as unfair.
    """
    row = [0.0] * 8

    # Check which risk types are mentioned anywhere in response
    for lid, lname in LABEL_MAP.items():
        if lname.upper() in response.upper():
            row[lid] = 1.0

    # If no risk types found and verdict is FAIR — return all zeros
    if sum(row) == 0:
        return row

    # Filter out false positives — if "None" appears right after risk type
    # and verdict is explicitly FAIR with no real risks
    response_upper = response.upper()
    if "VERDICT: FAIR" in response_upper and "RISK TYPE: NONE" in response_upper:
        return [0.0] * 8

    return row

# Load data
print("Loading test data...")
test_texts, test_labels = load_jsonl(DATA_DIR / "contract_test.jsonl", NUM_SAMPLES)
print(f"Running on {len(test_texts)} examples...")

preds  = []
labels = []

for i, (clause, label) in enumerate(zip(test_texts, test_labels)):
    if i % 25 == 0:
        print(f"  [{i}/{len(test_texts)}]...")

    prompt = f"[INST] {SYSTEM_PROMPT}\n\nClause:\n{clause[:600]} [/INST]"
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=700
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    preds.append(parse_response(response))
    labels.append(label)

preds  = np.array(preds)
labels = np.array(labels)

macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)

print(f"\n{'='*50}")
print("RESULTS — Mistral 7B zero-shot (prompt-only)")
print(f"{'='*50}")
print(f"Macro F1 : {macro_f1:.4f}")
print(f"Micro F1 : {micro_f1:.4f}")
print(classification_report(
    labels, preds,
    target_names=[LABEL_MAP[i] for i in range(8)],
    zero_division=0,
))

# Save results
with open(RESULTS_DIR / "results.md", "w") as f:
    f.write("# Baseline Results\n\n")
    f.write("| Model | Macro F1 |\n|---|---|\n")
    f.write("| Logistic Regression + TF-IDF | 0.1603 |\n")
    f.write("| BERT-base + weighted loss    | 0.5492 |\n")
    f.write(f"| Mistral 7B prompt-only       | {macro_f1:.4f} |\n")
    f.write("| Mistral 7B QLoRA fine-tuned  | TBD |\n")

import shutil
shutil.copy('/content/data/results/results.md',
            '/content/drive/MyDrive/contract_risk/results.md')
print("\nResults saved to Drive!")

Loading test data...
Running on 200 examples...
  [0/200]...
  [25/200]...
  [50/200]...
  [75/200]...
  [100/200]...
  [125/200]...
  [150/200]...
  [175/200]...

RESULTS — Mistral 7B zero-shot (prompt-only)
Macro F1 : 0.1285
Micro F1 : 0.1138
                         precision    recall  f1-score   support

Limitation of Liability       0.12      1.00      0.21         2
 Unilateral Termination       0.27      1.00      0.42         8
      Unilateral Change       0.04      0.50      0.07         2
        Content Removal       0.02      1.00      0.04         1
      Contract by Using       0.06      1.00      0.11         3
          Choice of Law       0.02      1.00      0.04         1
        Choice of Venue       0.02      1.00      0.04         1
     Forced Arbitration       0.05      1.00      0.09         2

              micro avg       0.06      0.95      0.11        20
              macro avg       0.07      0.94      0.13        20
           weighted avg       0.14    